In [ ]:
#include <iostream>
#include <vector>
#include <deque>
#include <list>
#include <algorithm>
#include <chrono>
#include <string>
#include <tuple>
#include <array>
#include <utility>
#include <type_traits>

# Задача 1

1.1 Сортировка

In [ ]:
template <typename RandomAccessIterator, typename Compare>
void my_sort(RandomAccessIterator first, RandomAccessIterator last, Compare comp) {
    if (std::distance(first, last) <= 1) return;

    auto pivot = *std::next(first, std::distance(first, last) / 2);
    RandomAccessIterator i = first;
    RandomAccessIterator j = last - 1;

    while (i <= j) {
        while (comp(*i, pivot)) i++;
        while (comp(pivot, *j)) j--;

        if (i <= j) {
            std::iter_swap(i, j);
            i++;
            j--;
        }
    }
    if (first < j) my_sort(first, j + 1, comp);
    if (i < last) my_sort(i, last, comp);
}

inline bool compare_func(int a, int b) {
    return a < b;
}

struct LessThan {
    bool operator()(int a, int b) const { return a < b; }
};

int main() {
    std::vector<int> v = {10, 5, 8, 1, 7};

    my_sort(v.begin(), v.end(), compare_func);

    my_sort(v.begin(), v.end(), LessThan());

    my_sort(v.begin(), v.end(), [](int a, int b) -> bool {
        return a < b;
    });

    return 0;
}

Замер времени

In [ ]:
void run_benchmarks() {
    const int N = 50000;
    std::vector<int> v(N);
    std::deque<int> d(N);

    for(int i = 0; i < N; ++i) {
        int val = rand();
        v[i] = d[i] = val;
    }

    auto start_v = std::chrono::high_resolution_clock::now();
    my_sort(v.begin(), v.end(), std::less<int>());
    auto end_v = std::chrono::high_resolution_clock::now();

    auto start_d = std::chrono::high_resolution_clock::now();
    my_sort(d.begin(), d.end(), std::less<int>());
    auto end_d = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double> diff_v = end_v - start_v;
    std::chrono::duration<double> diff_d = end_d - start_d;

    std::cout << "Vector: " << diff_v.count() << "s\n";
    std::cout << "Deque:  " << diff_d.count() << "s\n";
}

Для сортировки $O(n \ln n)$ нам нужны итераторы произвольного доступа. Без них мы не сможем за $O(1)$ прыгнуть в середину, чтобы pivot, и не сможем сравнивать итераторы через "или".

**Почему vector быстрее deque?**

Вектор - монолитный кусок памяти. Процессор один раз залез в память, забил кэш соседними числами и быстро их отсортировал. Дек - куча разных кусков. Процессору приходится постоянно перемещаться между ними, из-за чего кэш работает плохо и всё тормозит.

**Где не сработает?**

В std::list. У него итераторы умеют только ++ и --. Наша сортировка просто не скомпилируется, потому что мы не сможем сделать it + n или it1 < it2.

**Что можно передать в comp?**

Все, что можно вызвать как функцию: Обычную функцию, Функтор, Лямбду.

1.2 Адаптер над контейнером

In [ ]:
template <typename T, typename Container = std::deque<T>>
class MyStack {
protected:
    Container c;

public:
    void push(const T& val) { c.push_back(val); }
    void pop() { if (!c.empty()) c.pop_back(); }

    auto& top() { return c.back(); }
    const auto& top() const { return c.back(); }

    size_t size() const { return c.size(); }
    bool empty() const { return c.empty(); }
};

template <>
class MyStack<char, std::string> {
private:
    std::string c;

public:
    void push(char val) { c.push_back(val); }
    void pop() { if (!c.empty()) c.pop_back(); }

    char& top() { return c.back(); }

    size_t size() const { return c.length(); }
    bool empty() const { return c.empty(); }
};

int main() {
    MyStack<int> s_int;
    s_int.push(10);

    MyStack<double, std::vector<double>> s_vec;
    s_vec.push(3.14);

    MyStack<char, std::string> s_str;
    s_str.push('F');
    s_str.push('P'); // MIPT style

    std::cout << "Top: " << s_str.top() << ", Size: " << s_str.size() << std::endl;

    return 0;
}

# Задача 2

# 2 Вариабельные шаблоны

2.1 Рекурсивное инстанцирование

In [ ]:
template <size_t k, typename... Args>
struct TupleProcessor {
    static void process(const std::tuple<Args...>& t, auto& op) {
        if constexpr (k < sizeof...(Args)) {

            op(std::get<k>(t));

            TupleProcessor<k + 1, Args...>::process(t, op);
        }
    }
};

template <typename... Args, typename UnaryOperation>
void apply_to_all(const std::tuple<Args...>& t, UnaryOperation op) {
    TupleProcessor<0, Args...>::process(t, op);
}

int main() {
    auto my_tuple = std::make_tuple(42, 3.14, std::string("MIPT"), '!');

    auto print_op = [](const auto& x) {
        std::cout << x << " ";
    };

    std::cout << "Результат работы: ";
    apply_to_all(my_tuple, print_op);
    std::cout << std::endl;

    return 0;
}

# Задача 3

3.1 Декартово произведение

In [ ]:
template <typename... Arrays>
constexpr size_t get_total_size() {
    return (std::tuple_size_v<Arrays> * ...);
}

template <size_t LinearIdx, size_t ArrayIdx, typename... Arrays>
constexpr auto get_element_for_dim(const std::tuple<Arrays...>& arrays) {
    constexpr auto sizes = std::array<size_t, sizeof...(Arrays)>{std::tuple_size_v<Arrays>...};

    size_t stride = 1;
    for (size_t j = ArrayIdx + 1; j < sizeof...(Arrays); ++j) {
        stride *= sizes[j];
    }

    size_t element_idx = (LinearIdx / stride) % sizes[ArrayIdx];

    return std::get<element_idx>(std::get<ArrayIdx>(arrays));
}

template <typename... Arrays, size_t... LinearIs, size_t... ArrayIs>
constexpr auto cartesian_product_impl(
    const std::tuple<Arrays...>& arrays,
    std::index_sequence<LinearIs...>,
    std::index_sequence<ArrayIs...>)
{
    return std::array{
        std::make_tuple(
            get_element_for_dim<LinearIs, ArrayIs>(arrays)...
        )...
    };
}

template <typename... Arrays>
constexpr auto make_cartesian_product(const Arrays&... arrays) {
    constexpr size_t total_size = get_total_size<Arrays...>();
    return cartesian_product_impl(
        std::forward_as_tuple(arrays...),
        std::make_index_sequence<total_size>{},
        std::make_index_sequence<sizeof...(Arrays)>{}
    );
}

int main() {
    constexpr std::array<int, 2> arr1 = {1, 2};
    constexpr std::array<char, 3> arr2 = {'a', 'b', 'c'};

    constexpr auto result = make_cartesian_product(arr1, arr2);

    std::cout << "Cartesian Product size: " << result.size() << std::endl;
    std::cout << "Elements:" << std::endl;

    for (const auto& t : result) {
        std::cout << "( " << std::get<0>(t) << ", " << std::get<1>(t) << " )" << std::endl;
    }

    return 0;
}

3.2 Линейный рекуррент

In [ ]:
template <typename CoeffsSequence, typename InitSequence>
struct Recurrent;

template <int... As, int... Xs>
struct Recurrent<std::integer_sequence<int, As...>, std::integer_sequence<int, Xs...>> {

    static constexpr int next_val(int... last_n) {
        return ((As * last_n) + ...);
    }

    template <size_t K>
    struct Get {
        static constexpr size_t N = sizeof...(As);

        static constexpr int value = compute<K>();

    private:
        template <size_t CurrentK, int... CurrentXs>
        static constexpr int compute() {
            if constexpr (CurrentK < N) {
                constexpr int initial_values[] = {Xs...};
                return initial_values[CurrentK];
            } else {
                return Step<N, CurrentK, Xs...>::value;
            }
        }
    };
};

template <int... Coeffs>
struct LinearSequence {
    template <int... Inits>
    struct WithInits {

        template <size_t K>
        static constexpr int get() {
            constexpr int initial[] = {Inits...};
            constexpr int a[] = {Coeffs...};
            constexpr size_t N = sizeof...(Coeffs);

            if constexpr (K < N) {
                return initial[K];
            } else {
                return calculate<K>(std::make_index_sequence<N>{});
            }
        }

    private:
        template <size_t K, size_t... I>
        static constexpr int calculate(std::index_sequence<I...>) {
            constexpr size_t N = sizeof...(Coeffs);
            return ((Coeffs * WithInits::get<K - N + I>()) + ...);
        }
    };
};

int main() {
    using Fibonacci = LinearSequence<1, 1>::WithInits<0, 1>;

    using MyRec = LinearSequence<2, 0, 3>::WithInits<1, 1, 1>;

    std::cout << "Fib(10) = " << Fibonacci::get<10>() << std::endl;
    std::cout << "MyRec(5) = " << MyRec::get<5>() << std::endl;

    return 0;
}

# Задача 4

In [ ]:
template<typename Iterator>
void check_sort_compatibility() {
    using Category = typename std::iterator_traits<Iterator>::iterator_category;

    static_assert(std::is_base_of_v<std::random_access_iterator_tag, Category>,
        "ОШИБКА");
}

template<typename Iterator, typename Compare>
void sort_with_interface(Iterator first, Iterator last, Compare comp) {
    check_sort_compatibility<Iterator>();
    std::cout << "Проверка пройдена" << std::endl;
}

template<typename T, typename = void>
struct has_push_back : std::false_type {};

template<typename T>
struct has_push_back<T, std::void_t<decltype(std::declval<T>().push_back(std::declval<typename T::value_type>()))>>
    : std::true_type {};

template<typename Container>
struct MyStackAdapter {
    static_assert(has_push_back<Container>::value,
        "ОШИБКА!");
    Container c;
};

int main() {

    std::vector<int> v = {3, 1, 2};
    sort_with_interface(v.begin(), v.end(), std::less<int>());
    MyStackAdapter<std::vector<int>> good_stack;

    std::cout << "Вектор успешно прошел все проверки!" << std::endl;

    return 0;
}

